# Notebook 1: PPE Detection — Getting Started

**Phases 1–2** | ~35 minutes

In this notebook you will:
1. Verify your environment (GPU, packages, models)
2. Explore the synthetic PPE dataset
3. Run a zero-shot baseline with YOLOe-26n (open-vocabulary detection)
4. Auto-label the dataset using a foundation model (SAM3, Qwen3-VL, or Grounding DINO)
5. Inspect the generated labels

> **Tip**: Ask your Claude Code CV engineer partner for help at any point!
> Just describe what you want to do and it will guide you.

---
## 0. Environment Check

Run this cell first to verify everything is set up correctly.

In [ ]:
import sys, torch, platform
from pathlib import Path

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} ({torch.version.cuda})")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:   {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

import ultralytics
print(f"Ultralytics:  {ultralytics.__version__}")

import transformers
print(f"Transformers: {transformers.__version__}")

# Set paths relative to this notebook
WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"
IMAGES_DIR = DATA_DIR / "synthetic_ppe"

print(f"\nWorkshop dir: {WORKSHOP_DIR}")
print(f"Data dir:     {DATA_DIR}")
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"
print("\n✓ Environment OK")

---
## 1. Dataset Inventory

Our dataset: **91 synthetic construction site images** generated with Gemini 3 Pro,
spread across 11 scene categories:

| Category | Count | Description |
|----------|-------|-------------|
| `cctv` | 10 | Security camera angles |
| `mixed_compliance` | 10 | Mixed hard hat compliance |
| `edge_cases` | 10 | Challenging lighting/occlusion |
| `warehouse` | 10 | Indoor warehouse scenes |
| `highway` | 10 | Road construction |
| `highrise` | 10 | High-rise building sites |
| `easy` | 5 | Unambiguous, simple scenes |
| `close_up` | 12 | Close-up 2–4 workers |
| `ambiguous` | 14 | Deliberately hard cases |

Let's look at them.

In [ ]:
# ── Utility: Show image grid ─────────────────────────────────────────────────

import matplotlib.pyplot as plt
from PIL import Image
import random

def show_image_grid(image_paths, ncols=4, figsize=(16, 12), title=None):
    """Display a grid of images with their filenames."""
    nrows = (len(image_paths) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1:
        axes = [axes]
    axes = [ax for row in axes for ax in (row if hasattr(row, '__iter__') else [row])]
    
    for ax, path in zip(axes, image_paths):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(Path(path).parent.name + "/" + Path(path).stem, fontsize=8)
        ax.axis("off")
    for ax in axes[len(image_paths):]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


def count_labels(dataset_dir):
    """Count labels per class in a YOLO dataset directory."""
    import yaml
    dataset_dir = Path(dataset_dir)
    
    # Load class names
    yaml_path = dataset_dir / "dataset.yaml"
    class_names = {}
    if yaml_path.exists():
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        class_names = cfg.get("names", {})
    
    counts = {}
    for split in ["train", "val"]:
        labels_dir = dataset_dir / "labels" / split
        if not labels_dir.exists():
            continue
        split_counts = {}
        for lf in sorted(labels_dir.glob("*.txt")):
            for line in lf.read_text().strip().splitlines():
                cls_id = int(line.split()[0])
                name = class_names.get(cls_id, f"class_{cls_id}")
                split_counts[name] = split_counts.get(name, 0) + 1
        counts[split] = split_counts
    
    return counts

In [ ]:
# Inventory: count images per category
categories = sorted([d for d in IMAGES_DIR.iterdir() if d.is_dir()])
print(f"Total categories: {len(categories)}")
print()

all_images = []
for cat in categories:
    imgs = sorted(cat.glob("*.jpg")) + sorted(cat.glob("*.png")) + sorted(cat.glob("*.webp"))
    all_images.extend(imgs)
    print(f"  {cat.name:25s}  {len(imgs):3d} images")

print(f"\n  {'TOTAL':25s}  {len(all_images):3d} images")

In [ ]:
# Show a random sample of 8 images
random.seed(42)
sample = random.sample(all_images, min(8, len(all_images)))
show_image_grid(sample, ncols=4, title="Random Sample from Synthetic PPE Dataset")

---
## 2. Phase 1 — Zero-Shot Baseline with YOLOe-26n

Before training anything, let's see what an **open-vocabulary** detector can do
out of the box. YOLOe-26n can detect any concept you describe in text — no training needed.

We'll set it to detect `["hard hat", "person"]` and evaluate against our validation set.

### Tool Reference

```bash
# evaluate_yoloe_26n.py — zero-shot evaluation (no CLI args, hardcoded paths)
# Evaluates YOLOe-26n-seg on the exp_a_filtered dataset
python data/evaluate_yoloe_26n.py
```

**Questions to think about:**
- What mAP50 do you expect from a model that's never seen construction sites?
- What's the inference speed? Is this production-viable at 30+ FPS?

In [ ]:
# TODO: Run the zero-shot baseline
# Option A: Use the evaluation script
#   !python {DATA_DIR}/evaluate_yoloe_26n.py
#
# Option B: Quick inline test — load YOLOe and run on a few images
#   from ultralytics import YOLO
#   model = YOLO("yoloe-26n-seg.pt")
#   model.set_classes(["hard hat", "person"])
#   results = model(str(sample[0]))
#   results[0].show()
#
# Ask Claude Code: "Run the YOLOE zero-shot baseline and show me the results"


In [ ]:
# TODO: Visualize some zero-shot detections
# Try different images — which categories work well? Which fail?


### Phase 1 Observations

Write your observations here:
- mAP50: ___
- Inference speed: ___ ms/image
- Strengths: ___
- Weaknesses: ___

---
## 3. Phase 2 — Auto-Labeling with Foundation Models

The zero-shot model gives us a baseline, but we need a **fast, specialized** detector.
To train one, we need labels. Instead of hand-labeling 91 images, we'll use a
foundation model as an auto-labeler ("teacher").

### Available Auto-Labeling Tools

| Tool | Model | Classes | Speed | HF Access |
|------|-------|---------|-------|-----------|
| `auto_label_sam3_hf.py` | SAM3 (HuggingFace) | 2-class or 3-class | ~2-3 sec/image | **Gated** (needs approval) |
| `auto_label_qwen3_vl.py` | Qwen3-VL (VLM) | 2-class or 3-class | ~3-5 sec/image | **Ungated** (works immediately) |
| `auto_label_ppe_2class.py` | Grounding DINO | 2-class only | ~1-2 sec/image | Ungated |

### Option A: SAM3 with 2-class mode (`exp_a`) — if you have SAM3 access

```bash
python data/auto_label_sam3_hf.py \
    --mode exp_a \
    --source-dir data/synthetic_ppe \
    --output-dir data/ppe_dataset_sam3_exp_a \
    --device cuda
```

### Option B: Qwen3-VL — if SAM3 is not available (ungated, no approval needed)

```bash
python data/auto_label_qwen3_vl.py \
    --mode exp_a \
    --source-dir data/synthetic_ppe \
    --output-dir data/ppe_dataset_qwen3vl \
    --device cuda
```

> **Note**: Qwen3-VL uses a Vision-Language Model (VLM) to detect objects via grounding prompts.
> It produces the same YOLO-format output as SAM3, so all downstream tools work identically.
> The 8B model needs ~16GB VRAM. For less VRAM, add `--model-id Qwen/Qwen3-VL-4B-Instruct`.

### Option C: Grounding DINO — lightweight alternative

```bash
python data/auto_label_ppe_2class.py \
    --source-dir data/synthetic_ppe \
    --output-dir data/ppe_dataset_gdino \
    --device cuda
```

**Why 2-class (`exp_a`)?** The `hardhat` + `person` approach lets us derive compliance
through spatial post-processing ("does a hardhat overlap this person's head?").
The 3-class approach tries to detect `no_hardhat` — an absence, not a visible object.

In [ ]:
# TODO: Run auto-labeling
# Pick a labeling tool and run it. Ask Claude Code if you're unsure which to choose.
#
# Option A — SAM3 (if you have HuggingFace approval):
#   !python {DATA_DIR}/auto_label_sam3_hf.py --mode exp_a --device cuda
#
# Option B — Qwen3-VL (ungated, no approval needed):
#   !python {DATA_DIR}/auto_label_qwen3_vl.py --mode exp_a --device cuda
#
# Option C — Grounding DINO:
#   !python {DATA_DIR}/auto_label_ppe_2class.py --device cuda
#
# Ask Claude Code: "Auto-label my dataset using the best available model"


In [ ]:
# After labeling: count what we got
# Replace the path with your actual output directory
LABELED_DIR = DATA_DIR / "ppe_dataset_sam3_exp_a"  # adjust if needed

if LABELED_DIR.exists():
    counts = count_labels(LABELED_DIR)
    for split, split_counts in counts.items():
        print(f"\n{split}:")
        for name, count in sorted(split_counts.items()):
            print(f"  {name:15s}  {count:4d}")
else:
    print(f"Dataset not found at {LABELED_DIR}")
    print("Run the auto-labeling cell above first.")

---
## 4. Inspect the Labels

**Never train on labels you haven't looked at.** Let's visualize the auto-generated
bounding boxes overlaid on the images.

```bash
python data/visualize_gt_annotations.py \
    --dataset-dir data/ppe_dataset_sam3_exp_a \
    --output-dir data/label_viz \
    --split val
```

In [ ]:
# ── Utility: Show labels on images ───────────────────────────────────────────

import numpy as np

CLASS_COLORS = {
    0: (0, 200, 0),      # hardhat — green
    1: (220, 30, 30),     # no_hardhat or person — red
    2: (30, 100, 220),    # person (3-class) — blue
}

def show_labeled_image(image_path, label_path, class_names=None, figsize=(12, 8)):
    """Display an image with YOLO-format bounding box labels overlaid."""
    img = np.array(Image.open(image_path))
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)
    
    if Path(label_path).exists():
        for line in Path(label_path).read_text().strip().splitlines():
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            
            color = np.array(CLASS_COLORS.get(cls_id, (200, 200, 200))) / 255
            name = class_names.get(cls_id, f"class_{cls_id}") if class_names else f"class_{cls_id}"
            
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, name, color=color, fontsize=8, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))
    
    ax.set_title(Path(image_path).name)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO: Inspect labeled images
# Look at 5-10 validation images with their labels.
# What do you notice? Are all the labels correct?
#
# Ask Claude Code: "Show me the auto-labels on a few validation images"


### Phase 2 Observations

Write your observations here:
- How many labels were generated? ___
- Label quality impression (1-5): ___
- Common errors you noticed: ___
- Any tiny/spurious labels? ___

---

**Next**: Open `02_inspect_iterate_train.ipynb` to clean the labels and train a model.